# 311 — Agent Setup

Bootstrap for the 31x interactive investigation notebook series.

This notebook is `%run` at the top of every other 31x notebook.
It establishes:
- Environment variables and API clients
- Neo4j connection health check
- Combined MCP tool list (Neo4j MCP + FastMCP)
- `execute_tool()` dispatcher used by all agents
- Shared `GRAPH_SCHEMA_HINT` constant

**Prerequisites:** Neo4j AuraDB seeded with Layers 1 & 2 data (run notebooks 111–216 first).

In [1]:
import sys, os, json, logging
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
print('Environment loaded.')

Environment loaded.


## Step 1 — Neo4j connection + health check

In [2]:
from src.graph.connection import Neo4jConnection

conn = Neo4jConnection()
conn.connect()
print('Connected to Neo4j.')

# Health check — verify key node labels exist
label_counts = conn.run_query("""
    CALL apoc.meta.stats() YIELD labels
    RETURN labels
""")

# Fallback if APOC not available
if not label_counts:
    label_counts = conn.run_query("""
        MATCH (n)
        WITH labels(n) AS lbs
        UNWIND lbs AS label
        RETURN label, count(*) AS count
        ORDER BY label
    """)

print('\nNode counts in graph:')
for row in label_counts:
    print(f'  {row}')

# Verify key Layer 1 and Layer 2 nodes are present
for label in ['Borrower', 'LoanApplication', 'BankAccount', 'Transaction',
               'Regulation', 'Section', 'Requirement', 'Threshold', 'Chunk']:
    result = conn.run_query(f'MATCH (n:{label}) RETURN count(n) AS n')
    count = result[0]['n'] if result else 0
    status = '✓' if count > 0 else '✗ MISSING'
    print(f'  {status}  {label}: {count}')

2026-03-09 14:26:50,266 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Regulation': 3, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205


## Step 2 — API clients

In [3]:
import anthropic
from openai import OpenAI

claude_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
oai_client    = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL         = 'claude-sonnet-4-6'

print(f'Anthropic client ready. Model: {MODEL}')
print(f'OpenAI client ready.')

Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.


## Step 3 — Build combined MCP tool list

In [4]:
# -------------------------------------------------------------------
# Neo4j MCP tools (mcp-neo4j-cypher)
# These match the exact schema of the mcp-neo4j-cypher package.
# In production these are fetched from the MCP server manifest;
# here we define them inline for notebook portability.
# -------------------------------------------------------------------

NEO4J_MCP_TOOLS = [
    {
        'name': 'read-neo4j-cypher',
        'description': (
            'Execute a read-only Cypher query against the Neo4j graph database. '
            'Returns result rows as a list of dicts. '
            'YOU generate the Cypher — use the GRAPH_SCHEMA_HINT in the system prompt. '
            'Always include LIMIT (max 100). Never use MERGE/CREATE/DELETE/SET.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Valid read-only Cypher query with LIMIT clause.',
                },
                'params': {
                    'type': 'object',
                    'description': 'Optional parameter dict for parameterised queries.',
                    'default': {},
                },
            },
            'required': ['query'],
        },
    },
    {
        'name': 'write-neo4j-cypher',
        'description': (
            'Execute a write Cypher query (MERGE, CREATE, SET) against Neo4j. '
            'Use ONLY for Layer 3 Assessment/Finding/ReasoningStep writes. '
            'Prefer persist_assessment tool for structured Layer 3 writes.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Write Cypher query.'},
                'params': {'type': 'object', 'default': {}},
            },
            'required': ['query'],
        },
    },
]

# -------------------------------------------------------------------
# FastMCP tool definitions
# Mirrors the @mcp.tool() functions in src/mcp/investigation_server.py
# -------------------------------------------------------------------

FASTMCP_TOOLS = [
    {
        'name': 'traverse_compliance_path',
        'description': (
            'Cross-layer compliance traversal. '
            'Walks entity → Borrower → Jurisdiction (RESIDES_IN/REGISTERED_IN) '
            '→ Regulation (APPLIES_TO_JURISDICTION) → Section → Requirement → Threshold. '
            'Call this FIRST for any compliance question to get the full regulatory framework. '
            'Returns applicable thresholds for the entity jurisdiction and loan type.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'entity_id':    {'type': 'string', 'description': "e.g. 'LOAN-0002' or 'BRW-0001'"},
                'entity_type':  {'type': 'string', 'enum': ['LoanApplication', 'Borrower']},
                'regulation_id':{'type': 'string', 'description': "Optional: 'APS-112'|'APG-223'|'APS-220'", 'default': ''},
            },
            'required': ['entity_id', 'entity_type'],
        },
    },
    {
        'name': 'retrieve_regulatory_chunks',
        'description': (
            'Semantic similarity search over regulatory Chunk nodes using the '
            'chunk_embeddings Neo4j vector index (OpenAI text-embedding-3-small, cosine). '
            'Use to retrieve supporting regulation text when writing a finding.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'query_text':    {'type': 'string', 'description': 'Regulatory concept to search.'},
                'regulation_id': {'type': 'string', 'default': '', 'description': "Optional filter: 'APS-112'|'APG-223'|'APS-220'"},
                'top_k':         {'type': 'integer', 'default': 5, 'description': 'Number of chunks (max 20).'},
            },
            'required': ['query_text'],
        },
    },
    {
        'name': 'detect_graph_anomalies',
        'description': (
            'Run a named rule-based anomaly pattern against the graph. '
            "pattern_name values: 'transaction_structuring' (sub-$10k suspicious transfers, finds ACC-0596), "
            "'high_lvr_loans' (LVR>=90, finds LOAN-0002/LOAN-0013), "
            "'high_risk_industry' (gambling/fin-assets, finds BRW-0624/BRW-0627), "
            "'layered_ownership' (OWNS depth>=2, finds BRW-0582 chain), "
            "'high_risk_jurisdiction' (JUR-VU/JUR-MM/JUR-KH), "
            "'guarantor_concentration' (guarantor on 2+ loans)."
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'pattern_name': {
                    'type': 'string',
                    'enum': ['transaction_structuring', 'high_lvr_loans',
                             'high_risk_industry', 'layered_ownership',
                             'high_risk_jurisdiction', 'guarantor_concentration'],
                },
                'entity_id': {'type': 'string', 'default': '', 'description': 'Optional entity scope.'},
            },
            'required': ['pattern_name'],
        },
    },
    {
        'name': 'persist_assessment',
        'description': (
            'Persist a compliance Assessment with Findings and ReasoningSteps to Layer 3 (Neo4j). '
            'Idempotent MERGE. Call after completing compliance analysis to store reasoning.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'entity_id':       {'type': 'string'},
                'entity_type':     {'type': 'string', 'enum': ['LoanApplication', 'Borrower']},
                'regulation_id':   {'type': 'string'},
                'verdict':         {'type': 'string', 'enum': ['COMPLIANT','NON_COMPLIANT','REQUIRES_REVIEW','ANOMALY_DETECTED','INFORMATIONAL']},
                'confidence':      {'type': 'number', 'minimum': 0, 'maximum': 1},
                'findings':        {'type': 'array', 'items': {'type': 'object'}},
                'reasoning_steps': {'type': 'array', 'items': {'type': 'object'}},
                'agent':           {'type': 'string', 'default': 'compliance_agent'},
            },
            'required': ['entity_id', 'entity_type', 'regulation_id', 'verdict', 'confidence'],
        },
    },
    {
        'name': 'trace_evidence',
        'description': (
            'Walk a stored Assessment back to all cited regulatory nodes. '
            'Returns findings, reasoning steps, cited sections (with text), '
            'and cited chunks (with text excerpt). '
            "Use when asked 'why was this flagged?' or 'show your reasoning'."
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'assessment_id': {'type': 'string', 'description': "e.g. 'ASSESS-LOAN-0002-APG-223-2026-03-09'"},
            },
            'required': ['assessment_id'],
        },
    },
]

TOOLS = NEO4J_MCP_TOOLS + FASTMCP_TOOLS
print(f'Tool registry: {len(NEO4J_MCP_TOOLS)} Neo4j MCP + {len(FASTMCP_TOOLS)} FastMCP = {len(TOOLS)} total')
for t in TOOLS:
    print(f'  {t["name"]}')

Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrieve_regulatory_chunks
  detect_graph_anomalies
  persist_assessment
  trace_evidence


## Step 4 — execute_tool dispatcher

In [ ]:
import re
from src.mcp.tools_impl import (
    traverse_compliance_path,
    retrieve_regulatory_chunks,
    detect_graph_anomalies,
    persist_assessment,
    trace_evidence,
)

_WRITE_KEYWORDS = {'MERGE', 'CREATE', 'DELETE', 'SET', 'DETACH'}

def execute_tool(tool_name: str, tool_input: dict) -> dict:
    """
    Dispatch a Claude tool_use call to the correct handler.

    Neo4j MCP tools execute Cypher via the shared conn.
    FastMCP tools call tools_impl functions directly (plain callables).
    """
    import logging
    logger = logging.getLogger('execute_tool')
    logger.info('Tool: %s | inputs: %s', tool_name, list(tool_input.keys()))

    try:
        # ── Neo4j MCP ────────────────────────────────────────────
        if tool_name == 'read-neo4j-cypher':
            query  = tool_input.get('query', '')
            params = tool_input.get('params', {})
            # Safety guard — reject write operations (whole-word match to avoid
            # false positives like ASSESSMENT containing SET)
            query_words = set(re.findall(r'\b[A-Z]+\b', query.upper()))
            if query_words & _WRITE_KEYWORDS:
                return {'error': 'read-neo4j-cypher does not allow write operations.'}
            return {'rows': conn.run_query(query, params)}

        elif tool_name == 'write-neo4j-cypher':
            query  = tool_input.get('query', '')
            params = tool_input.get('params', {})
            return {'rows': conn.run_query(query, params)}

        # ── FastMCP ───────────────────────────────────────────────
        elif tool_name == 'traverse_compliance_path':
            return traverse_compliance_path(**tool_input)

        elif tool_name == 'retrieve_regulatory_chunks':
            return retrieve_regulatory_chunks(**tool_input)

        elif tool_name == 'detect_graph_anomalies':
            return detect_graph_anomalies(**tool_input)

        elif tool_name == 'persist_assessment':
            return persist_assessment(**tool_input)

        elif tool_name == 'trace_evidence':
            return trace_evidence(**tool_input)

        else:
            return {'error': f'Unknown tool: {tool_name}'}

    except Exception as e:
        logger.error('Tool %s failed: %s', tool_name, e, exc_info=True)
        return {'error': str(e)}

print('execute_tool dispatcher ready.')

## Step 5 — Schema constant

In [6]:
from src.mcp.schema import GRAPH_SCHEMA_HINT
print(f'GRAPH_SCHEMA_HINT loaded: {len(GRAPH_SCHEMA_HINT)} chars')

GRAPH_SCHEMA_HINT loaded: 7045 chars


## Setup complete

Exported: `conn`, `claude_client`, `oai_client`, `MODEL`, `TOOLS`, `execute_tool`, `GRAPH_SCHEMA_HINT`

Next: Run `notebooks/312_graph_tools.ipynb` to test all tools.